In [1]:
import piplite
await piplite.install(["pandas", "scikit-learn", "ipywidgets"])
print("Packages ready. Run the next cell.")

Packages ready. Run the next cell.


In [3]:
import io
import base64
import numpy as np
import pandas as pd
import ipywidgets as widgets
from IPython.display import display, HTML, Markdown, clear_output
from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import train_test_split
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.naive_bayes import GaussianNB
from sklearn.svm import SVC
from sklearn.metrics import (accuracy_score, roc_auc_score, precision_score,
    recall_score, f1_score, matthews_corrcoef, confusion_matrix, classification_report)

# Train once on training rows; uploaded CSVs are used only for evaluation.
dataset = load_breast_cancer(as_frame=True)
FEATURES = list(dataset.feature_names)
LABELS = {0: "malignant", 1: "benign"}
X_train, X_test, y_train, y_test = train_test_split(
    dataset.data, dataset.target, test_size=0.2, random_state=42,
    stratify=dataset.target)
models = {
    "Logistic Regression": make_pipeline(StandardScaler(), LogisticRegression(max_iter=2000, random_state=42)),
    "Decision Tree": DecisionTreeClassifier(random_state=42),
    "KNN": make_pipeline(StandardScaler(), KNeighborsClassifier()),
    "Gaussian Naive Bayes": GaussianNB(),
    "Support Vector Machine": make_pipeline(StandardScaler(), SVC(probability=True, random_state=42)),
}
for model in models.values():
    model.fit(X_train, y_train)
builtin_test = X_test.copy()
builtin_test["target"] = y_test
current_test = builtin_test.copy()

chooser = widgets.Dropdown(options=list(models), description="Classifier:",
    style={"description_width": "initial"}, layout=widgets.Layout(width="480px"))
upload = widgets.FileUpload(accept=".csv", multiple=False, description="Upload test CSV")
reset = widgets.Button(description="Use built-in test data", layout=widgets.Layout(width="200px"))
output = widgets.Output()

def download_link(frame, filename, title):
    payload = base64.b64encode(frame.to_csv(index=False).encode()).decode()
    return HTML(f'<a download="{filename}" href="data:text/csv;base64,{payload}">{title}</a>')

def render(change=None):
    with output:
        clear_output(wait=True)
        try:
            df = current_test.copy()
            missing = [c for c in FEATURES + ["target"] if c not in df.columns]
            if missing:
                raise ValueError(f"Missing required columns: {missing}")
            if df.empty:
                raise ValueError("The CSV has no rows.")
            X = df[FEATURES].apply(pd.to_numeric, errors="raise")
            y = pd.to_numeric(df["target"], errors="raise")
            if not y.isin([0, 1]).all():
                raise ValueError("Target values must be 0 (malignant) or 1 (benign).")
            if not np.isfinite(X.to_numpy()).all():
                raise ValueError("Features cannot contain missing or infinite values.")
            y = y.astype(int)
            model = models[chooser.value]
            pred = model.predict(X)
            prob = model.predict_proba(X)[:, list(model.classes_).index(1)]
            metrics = {
                "Accuracy": accuracy_score(y, pred),
                "AUC": roc_auc_score(y, prob) if y.nunique() == 2 else np.nan,
                "Precision": precision_score(y, pred, zero_division=0),
                "Recall": recall_score(y, pred, zero_division=0),
                "F1 Score": f1_score(y, pred, zero_division=0),
                "MCC": matthews_corrcoef(y, pred),
            }
            display(Markdown(f"## Results: {chooser.value}"))
            display(Markdown(f"**{len(df)} test rows** | 0 = malignant, 1 = benign | Positive class: benign"))
            display(pd.DataFrame([metrics]).round(4))
            display(Markdown("### Confusion matrix"))
            display(pd.DataFrame(confusion_matrix(y, pred, labels=[0, 1]),
                index=["Actual malignant", "Actual benign"],
                columns=["Predicted malignant", "Predicted benign"]))
            display(Markdown("### Classification report"))
            print(classification_report(y, pred, labels=[0, 1],
                target_names=["malignant", "benign"], zero_division=0))
            result = df.copy()
            result["predicted_target"] = pred
            result["predicted_label"] = [LABELS[int(v)] for v in pred]
            display(Markdown("### Input rows and predictions"))
            # Put predictions first so they are visible without horizontal scrolling.
            columns = ["target", "predicted_target", "predicted_label"] + FEATURES
            display(HTML('<div style="max-height:360px;overflow:auto">' +
                result[columns].to_html(index=False) + '</div>'))
            display(download_link(result, "predictions.csv", "Download predictions CSV"))
            display(download_link(builtin_test, "test_data.csv", "Download built-in test CSV"))
        except Exception as error:
            display(Markdown(f"**Could not evaluate CSV:** {error}"))

def on_upload(change):
    global current_test
    if not upload.value:
        return
    try:
        entry = next(iter(upload.value.values())) if isinstance(upload.value, dict) else upload.value[0]
        current_test = pd.read_csv(io.BytesIO(bytes(entry["content"])))
        render()
    except Exception as error:
        with output:
            clear_output(wait=True)
            print(f"Could not read CSV: {error}")

def use_builtin(button):
    global current_test
    current_test = builtin_test.copy()
    render()

chooser.observe(render, names="value")
upload.observe(on_upload, names="value")
reset.on_click(use_builtin)
display(Markdown("# Machine Learning Classification Model Explorer"))
display(Markdown("Wisconsin Diagnostic Breast Cancer dataset | Five trained classifiers"))
display(Markdown("Models trained once on **455 training rows**. Results below use **114 held-out test rows**. Uploading a CSV does not retrain the models."))
display(chooser, widgets.HBox([upload, reset]), output)
render()

# Machine Learning Classification Model Explorer

Wisconsin Diagnostic Breast Cancer dataset | Five trained classifiers

Models trained once on **455 training rows**. Results below use **114 held-out test rows**. Uploading a CSV does not retrain the models.

Dropdown(description='Classifier:', layout=Layout(width='480px'), options=('Logistic Regression', 'Decision Tr…

Output()